In [58]:
import torch
import torch.nn as nn 
import math

class lstm(nn.Module):
  def __init__(self,input_layer,hidden_layer):
    super().__init__()
    self.input=input_layer
    self.hidden=hidden_layer

    self.w=nn.Parameter(torch.Tensor(input_layer,hidden_layer*4))
    self.u=nn.Parameter(torch.Tensor(hidden_layer,hidden_layer*4))

    self.b=nn.Parameter(torch.Tensor(hidden_layer*4))

    self._init_weights()

  def _init_weights(self):
    
    stdv=1.0/math.sqrt(self.hidden)
    for weight in self.parameters():
      nn.init.uniform_(weight,-stdv,stdv)

  def forward(self,x,init_states=None):
    batch_size,seq,_=x.size()

    if init_states is None:
      h_t=torch.zeros(batch_size,self.hidden,device=x.device)
      c_t=torch.zeros(batch_size,self.hidden,device=x.device)

    else:
      h_t,c_t=init_states
    

    hseq=[]


    for t in range(seq):
      x_t=x[:,t,:]


      gates=torch.matmul(x_t,self.w)+torch.matmul(h_t,self.u)+self.b
      
      fgate,igate,cgate,ogate=gates.chunk(4,dim=1)


      fgate=torch.sigmoid(fgate)
      cgate=torch.tanh(cgate)
      igate=torch.sigmoid(igate)
      ogate=torch.sigmoid(ogate)

      c_t=c_t*fgate+cgate*igate

      h_t=ogate*torch.tanh(c_t)

      hseq.append(h_t.unsqueeze(1))

    torch.cat (hseq,dim=1) 
   
    return hseq,(c_t,h_t)
  


      



In [59]:
import pandas as pd
from torch.utils.data import Dataset,DataLoader
url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
df = pd.read_csv(url, compression='zip')
df = df[5::6].reset_index(drop=True)

req_column=['p (mbar)', 'T (degC)', 'rh (%)']
data=df[req_column].values
# print(data)

mean=data.mean(axis=0)
std=data.mean(axis=0)
normalize=(data-mean)/std


class weather(Dataset):
  def __init__(self,data,seqlen=72,pred=12):
    super().__init__()
    self.data=torch.tensor(data,dtype=torch.float32)
    self.seqlen=seqlen
    self.pred=pred

  def __len__(self):
    return  len(self.data)-self.seqlen-self.pred+1
  
  def __getitem__(self, index):
    
    x=self.data[index:index+self.seqlen]
    y = self.data[index + self.seqlen : index + self.seqlen + self.pred, 1]

    return x,y




In [ ]:

split=int(len(normalize)*0.8)
traindata=normalize[:split]
testdata=normalize[split:]


traindataset=weather(traindata,seqlen=72,pred=12)
testdataset=weather(testdata,seqlen=72,pred=12)




traindataloader=DataLoader(traindataset,batch_size=64,shuffle=True)
testtdataloader=DataLoader(testdataset,batch_size=64,shuffle=False)

class weatherpredict(nn.Module):
  def __init__(self,input,hidden=64,output=12):
    super().__init__()
    self.lstm=lstm(input,hidden)
    self.fi=nn.Linear(hidden,output)
  
  def forward(self,x):
    _,(ht,_)=self.lstm(x)
    y=self.fi(ht)
    return y

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model=weatherpredict(input=3,hidden=64,output=12).to(device)

criterion=nn.MSELoss()
optimiser=torch.optim.Adam(model.parameters(),lr=0.001)

epoch=10


for i in range(epoch):
  model.train()
  trainloss=0

  for batchx,batchy in traindataloader:

    batchx,batchy=batchx.to(device),batchy.to(device)

    ypred=model(batchx)

    loss=criterion(ypred,batchy)
    optimiser.zero_grad()
    loss.backward()

    nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)

    

    optimiser.step()

    trainloss+=loss.item() * batchx.size(0)
  trainlosss=trainloss/len(traindataset)
  print(f"Loss:trainloss{trainlosss}") 

  # --- Validation / Testing Loop ---
model.eval()  # Set model to evaluation mode
testloss = 0

with torch.no_grad():  # Turn off gradient calculation
    for batchx, batchy in testtdataloader:
        batchx, batchy = batchx.to(device), batchy.to(device)

        # Forward pass
        ypred = model(batchx)

        # Calculate loss
        loss = criterion(ypred, batchy)

        # Accumulate total test loss
        testloss += loss.item() * batchx.size(0)

# Calculate average loss over the whole test dataset
testlosss = testloss / len(testdataset)
print(f"Loss: testloss {testlosss:.4f}")  





KeyboardInterrupt: 

In [ ]:
print(torch.Tensor(5,4))

tensor([[-5.3337e+13,  1.8988e-42,  1.5392e-34, -9.1513e-01],
        [-3.7259e+20, -9.2257e-01, -2.1956e+34, -9.2904e-01],
        [-2.5161e+08, -9.3812e-01, -8.7513e-08, -9.3990e-01],
        [ 3.9036e-19, -9.4362e-01, -8.8415e+24, -9.4589e-01],
        [ 1.2365e+08, -9.4702e-01,  4.0827e+13, -9.4961e-01]])
